# AffinityMD — Romance Simulation (Notebook)

Interactive notebook for running the `renai_simu` package.

**Physics.** Particles (M/F, ages 20–30) diffuse by overdamped Langevin dynamics.
A Morse potential between any pair has an effective well depth
$\varepsilon_{ij}^{\text{eff}} = \varepsilon_0 \cdot \chi_{ij} \cdot B_{ij} \cdot f(\Delta\text{age}) \cdot P_{ij}$,
where $B_{ij}$ is a dynamic relationship strength and $P_{ij}$ is a field distortion from third parties.
Stochastic events (cheating / 환승 / 대쉬) probabilistically boost $B$ for non-coupled pairs and damage existing bonds.

Outputs are written to `outputs/notebook_run_<timestamp>/` so they never overwrite terminal runs.

## 1. Imports and setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from datetime import datetime

from renai_simu import Simulation, DEFAULT_PARAMS, visualize, io_utils

import matplotlib.pyplot as plt
%matplotlib inline
print('Ready.')

## 2. Parameters

Edit the overrides below. Anything not overridden uses `DEFAULT_PARAMS`.

In [ ]:
params_override = {
    'N_male':    8,
    'N_female':  8,
    'T_total':   150.0,
    'dt':        0.10,
    'seed':      42,

    # try tweaking these:
    # 'chi_het':   7.0,
    # 'kT':        2.0,
    # 'p_event':   0.003,
    # 'Q_perturb': 2.5,
}

# Unique output directory — separated from terminal runs by design.
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = Path('outputs') / f'notebook_run_{stamp}'
out_dir.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {out_dir}')

## 3. Run simulation

In [ ]:
sim = Simulation(params_override)
sim.run(verbose=True)

print()
n_c = sum(1 for i in sim.males if sim.couple[i] != -1)
print(f'Final couples : {n_c}/{min(sim.N_m, sim.N_f)}')
print(f'Total events  : {len(sim.event_log)}')
for i in sim.males:
    j = int(sim.couple[i])
    if j != -1:
        print(f'  {sim.labels[i]} <3 {sim.labels[j]}  (B={sim.B[i, j]:.3f})')

## 4. Visualize inline

In [ ]:
fig = visualize.plot_time_evolution(sim)
plt.show()

In [ ]:
fig = visualize.plot_relationship_graph(sim)
plt.show()

In [ ]:
fig = visualize.plot_B_heatmap(sim)
plt.show()

In [ ]:
fig = visualize.plot_snapshots(sim, n_snapshots=4)
plt.show()

## 5. Save all outputs

Writes figures + `parmsdict.txt` + `event_log.tsv` + `summary.json` into the notebook's run folder.

In [ ]:
visualize.plot_all(sim, out_dir)
io_utils.save_all(sim, out_dir)

print(f'All outputs saved to: {out_dir}')
for f in sorted(out_dir.iterdir()):
    print(' ', f.name)

## 6. (Optional) Inspect raw data

`sim.history` holds all recorded trajectories. `sim.event_log` is the list of stochastic events.

In [ ]:
print('Recorded time points:', len(sim.history['time']))
print('Total events       :', len(sim.event_log))

print('\nFirst 10 events:')
for ev in sim.event_log[:10]:
    t, i, j, kind = ev
    print(f'  t={t:6.2f}  {sim.labels[i]} x {sim.labels[j]}  [{kind}]')